# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mvdu12/ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/Mvdu12/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

numeric_feats = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
cat_feats = ['content_type', 'main_intent', 'competition_level']

X = df[numeric_feats + cat_feats].copy()
for c in numeric_feats:
    # missingness follows content_type (data dictionary) -- a flag column instead
    # of a blind fillna(0), so "not measured" doesn't get silently read as "zero".
    X[c + '_missing'] = X[c].isna().astype(int)
    X[c] = X[c].fillna(0)
X = pd.get_dummies(X, columns=cat_feats, dummy_na=True)

print(f"Feature vector: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"Label base rate (is_declining_label): {df['is_declining_label'].mean():.3f}")
X.head()


Feature vector: 30,000 rows x 45 columns
Label base rate (is_declining_label): 0.542


,search_volume,competition,cpc,word_count,char_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,...,content_type_nan,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_nan,competition_level_HIGH,competition_level_LOW,competition_level_MEDIUM,competition_level_nan
0,10.0,0.67,2.05,3221.0,20457.0,187,20,3803,29,17,...,False,False,False,False,True,False,True,False,False,False
1,90.0,0.01,0.05,2481.0,15562.0,445,25,15320,7,9,...,False,False,True,False,False,False,False,True,False,False
2,0.0,0.00,0.00,3515.0,23643.0,141,20,12581,11,11,...,False,False,True,False,False,False,False,True,False,False
3,10.0,0.00,0.00,0.0,0.0,463,22,11751,58,78,...,False,True,False,False,False,False,False,True,False,False
4,0.0,0.00,0.00,2803.0,17469.0,263,14,19140,24,145,...,False,False,True,False,False,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
feature_notes = pd.DataFrame([
    ("search_volume",           "keyword-level search demand estimate",         "blank -> 0 + *_missing flag (no keyword data on some content_types)", "yes -- set when keyword is targeted, before any traffic is observed"),
    ("competition",             "keyword competition score, 0-1",                "blank -> 0 + *_missing flag",                                          "yes -- same as above"),
    ("cpc",                     "cost-per-click estimate for the keyword",       "blank -> 0 + *_missing flag",                                          "yes -- same as above"),
    ("word_count",               "article word count",                            "blank -> 0 + *_missing flag (~26% missing, varies by content_type)",  "yes -- fixed once published"),
    ("char_count",               "article character count",                       "blank alongside word_count -> 0 + *_missing flag",                     "yes -- fixed once published"),
    ("content_age_days",         "days since content was created",                "not missing in this slice",                                            "yes -- a clock, not an outcome"),
    ("days_since_last_update",   "days since content was last touched",           "not missing in this slice",                                            "yes -- known the moment you'd act"),
    ("impressions_90d",         "trailing 90-day GSC impressions",               "not missing",                                                          "yes -- but it's a TRAILING window, not a future one"),
    ("clicks_90d",               "trailing 90-day GSC clicks",                    "not missing",                                                          "yes -- same trailing window"),
    ("sessions_90d",             "trailing 90-day GA4 sessions",                  "not missing",                                                          "yes -- same trailing window"),
    ("engaged_sessions_90d",     "trailing 90-day GA4 engaged sessions",          "not missing",                                                          "yes -- same trailing window"),
    ("ctr",                     "clicks_90d / impressions_90d x 100",             "not missing (0 when no clicks)",                                       "yes -- derived from the same trailing window"),
    ("avg_position",            "mean GSC position; 0 = no position data",       "0 means 'no data', kept as-is + relies on the 90d window",             "yes -- trailing window, not future"),
    ("engagement_rate",         "engaged_sessions_90d / sessions_90d x 100",     "not missing",                                                          "yes -- same trailing window"),
    ("scroll_rate",              "scroll_events_90d / pageviews_90d x 100, can exceed 100", "blank when pageviews_90d = 0",                             "yes -- same trailing window"),
    ("ai_traffic_pct",           "ai_sessions_90d / sessions_90d x 100, can exceed 100", "not missing",                                                    "yes -- same trailing window"),
    ("content_type",             "keyword / feedly / comparison article",         "no blanks",                                                            "yes -- fixed at creation"),
    ("main_intent",              "informational / transactional / commercial / navigational", "blank -> 'unknown' via dummy_na",                          "yes -- fixed at creation"),
    ("competition_level",        "LOW / MEDIUM / HIGH keyword competition",       "blank -> 'unknown' via dummy_na",                                      "yes -- fixed with the keyword"),
], columns=["feature", "meaning", "missing handling", "available before the label moment?"])

pd.set_option("display.max_colwidth", None)
feature_notes


,feature,meaning,missing handling,available before the label moment?
0,search_volume,keyword-level search demand estimate,blank -> 0 + *_missing flag (no keyword data on some content_types),"yes -- set when keyword is targeted, before any traffic is observed"
1,competition,"keyword competition score, 0-1",blank -> 0 + *_missing flag,yes -- same as above
2,cpc,cost-per-click estimate for the keyword,blank -> 0 + *_missing flag,yes -- same as above
3,word_count,article word count,"blank -> 0 + *_missing flag (~26% missing, varies by content_type)",yes -- fixed once published
4,char_count,article character count,blank alongside word_count -> 0 + *_missing flag,yes -- fixed once published
5,content_age_days,days since content was created,not missing in this slice,"yes -- a clock, not an outcome"
6,days_since_last_update,days since content was last touched,not missing in this slice,yes -- known the moment you'd act
7,impressions_90d,trailing 90-day GSC impressions,not missing,"yes -- but it's a TRAILING window, not a future one"
8,clicks_90d,trailing 90-day GSC clicks,not missing,yes -- same trailing window
9,sessions_90d,trailing 90-day GA4 sessions,not missing,yes -- same trailing window


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

y = df['is_declining_label']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))

def fit_eval(Xd, name):
    Xtr, Xte = Xd.iloc[tr_idx].copy(), Xd.iloc[te_idx].copy()
    ytr, yte = y.iloc[tr_idx], y.iloc[te_idx]
    num_cols = [c for c in Xd.columns if c in numeric_feats]
    sc = StandardScaler()
    Xtr[num_cols] = sc.fit_transform(Xtr[num_cols])
    Xte[num_cols] = sc.transform(Xte[num_cols])
    m = LogisticRegression(max_iter=1000, random_state=42)
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, p)
    print(f"{name}: AUC = {auc:.4f}")
    return auc

print("--- Test 1: label-derived feature (leakage taxonomy #1) ---")
auc_honest = fit_eval(X, "WITHOUT suspect (my honest feature set)")

X_leak_pct = X.copy()
X_leak_pct['trend_pct'] = df['trend_pct'].fillna(0)
auc_leak_pct = fit_eval(X_leak_pct, "WITH trend_pct added as a 'feature'")

X_leak_dir = X.copy()
X_leak_dir['trend_direction_down'] = (df['trend_direction'] == 'down').astype(int)
auc_leak_dir = fit_eval(X_leak_dir, "WITH trend_direction=='down' added (literally the label)")

print(
    f"\nVerdict: honest AUC {auc_honest:.3f} collapses to {auc_leak_pct:.3f} and "
    f"{auc_leak_dir:.3f} the moment trend_pct / trend_direction sneak in as features "
    f"-- the textbook confession of leakage taxonomy #1 (label-derived feature). "
    f"This confirms trend_direction/trend_pct must stay excluded (Section 4)."
)

print("\n--- Test 2: future/overlapping windows (leakage taxonomy #2) ---")
print(
    "All numeric features come from the SAME trailing-90-day window as the label "
    "(trend_direction is computed from last-30d vs prev-30d impressions, both inside "
    "that same 90-day snapshot). There is no separate 'future' window available in this "
    "starter CSV to leak from -- the real risk shows up later, in the full warehouse "
    "release, where fact_content_query_90d's window can overlap a label defined on the "
    "final month (flagged in the data dictionary). Noting it here so it isn't forgotten "
    "once I move to that release."
)

print("\n--- Test 3: decision-derived / product-flag features (leakage taxonomy #3) ---")
print(
    "FlyRank's own product outputs (health_score, priority_score, action_type, refresh "
    "flags) are not columns in this dataset at all, so there's no way to accidentally "
    "include them here -- but noting the rule for later: if they ever show up in the "
    "warehouse release, they are a baseline to beat, never a feature (using them would "
    "just be learning FlyRank's existing rule back)."
)

print("\n--- Test 4: split honesty -- random vs grouped (does the split itself leak?) ---")
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_cols = numeric_feats
sc = StandardScaler()
Xtr_r2, Xte_r2 = Xtr_r.copy(), Xte_r.copy()
Xtr_r2[num_cols] = sc.fit_transform(Xtr_r[num_cols])
Xte_r2[num_cols] = sc.transform(Xte_r[num_cols])
m_r = LogisticRegression(max_iter=1000, random_state=42)
m_r.fit(Xtr_r2, ytr_r)
p_r = m_r.predict_proba(Xte_r2)[:, 1]
auc_random = roc_auc_score(yte_r, p_r)
print(f"Random row-level split AUC: {auc_random:.4f}")
print(f"Grouped (client_id) split AUC: {auc_honest:.4f}")
print(
    f"\nGap: {auc_random - auc_honest:.3f}. The random split reports a higher score "
    f"because some of each client's pages leak into both train and test -- part of "
    f"what looks like 'skill' is really the model recognizing which client a page "
    f"belongs to. The grouped-split number is the one I trust and carry forward."
)


--- Test 1: label-derived feature (leakage taxonomy #1) ---
WITHOUT suspect (my honest feature set): AUC = 0.5395
WITH trend_pct added as a 'feature': AUC = 1.0000
WITH trend_direction=='down' added (literally the label): AUC = 1.0000

Verdict: honest AUC 0.539 collapses to 1.000 and 1.000 the moment trend_pct / trend_direction sneak in as features -- the textbook confession of leakage taxonomy #1 (label-derived feature). This confirms trend_direction/trend_pct must stay excluded (Section 4).

--- Test 2: future/overlapping windows (leakage taxonomy #2) ---
All numeric features come from the SAME trailing-90-day window as the label (trend_direction is computed from last-30d vs prev-30d impressions, both inside that same 90-day snapshot). There is no separate 'future' window available in this starter CSV to leak from -- the real risk shows up later, in the full warehouse release, where fact_content_query_90d's window can overlap a label defined on the final month (flagged in the data di

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
excluded = pd.DataFrame([
    ("trend_direction",  "IS the label (is_declining_label = trend_direction == 'down'). Test above shows AUC -> 1.0 the instant it's included."),
    ("trend_pct",        "defines trend_direction; same test above shows the identical AUC -> 1.0 collapse."),
    ("content_id",       "pseudonymous row identifier -- has no predictive meaning, only usable for joins."),
    ("client_id",        "pseudonymous client identifier -- used only for the grouped split (Section 3 / Test 4), never as a model input."),
    ("provider_used",    "flagged 'not a model feature' in the data dictionary -- which LLM wrote the page describes production process, not content quality or decline risk."),
    ("model_used",       "same reasoning as provider_used -- production metadata, not a content signal."),
    ("age_tier / freshness_tier / word_count_tier / char_count_tier / impression_tier / position_tier", "these are just bucketed versions of numeric columns already in X (content_age_days, days_since_last_update, word_count, char_count, impressions_90d, avg_position) -- including both would double-count the same information."),
    ("health_score / priority_score / action_type / refresh flags", "FlyRank's own product outputs -- not present in this starter CSV, and would be circular (decision-derived, leakage taxonomy #3) if they ever showed up in the warehouse release."),
], columns=["excluded field(s)", "why"])

pd.set_option("display.max_colwidth", None)
excluded


,excluded field(s),why
0,trend_direction,IS the label (is_declining_label = trend_direction == 'down'). Test above shows AUC -> 1.0 the instant it's included.
1,trend_pct,defines trend_direction; same test above shows the identical AUC -> 1.0 collapse.
2,content_id,"pseudonymous row identifier -- has no predictive meaning, only usable for joins."
3,client_id,"pseudonymous client identifier -- used only for the grouped split (Section 3 / Test 4), never as a model input."
4,provider_used,"flagged 'not a model feature' in the data dictionary -- which LLM wrote the page describes production process, not content quality or decline risk."
5,model_used,"same reasoning as provider_used -- production metadata, not a content signal."
6,age_tier / freshness_tier / word_count_tier / char_count_tier / impression_tier / position_tier,"these are just bucketed versions of numeric columns already in X (content_age_days, days_since_last_update, word_count, char_count, impressions_90d, avg_position) -- including both would double-count the same information."
7,health_score / priority_score / action_type / refresh flags,"FlyRank's own product outputs -- not present in this starter CSV, and would be circular (decision-derived, leakage taxonomy #3) if they ever showed up in the warehouse release."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.